# Hybrid: Clustering + Classification (Leakage-Safe)

Pipeline following requested diagram with train/test split before clustering, KMeans fitted on TRAIN only, cluster as feature, SMOTE on TRAIN, classifier trained on resampled TRAIN, evaluation on TEST.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score, precision_score,
    recall_score, f1_score
)

sys.path.append(os.path.abspath('..'))
from src import loader

In [ ]:
df = loader()
df['Churn'] = df['Churn'].astype(str)
df.head()

In [ ]:
TARGET = 'Churn'
X = df.drop(columns=[TARGET])
y = df[TARGET].replace({'Yes': 1, 'No': 0}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(pd.Series(y_train).value_counts(), pd.Series(y_test).value_counts())

In [ ]:
NUMERICAL_COLUMNS = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_COLUMNS = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                        'StreamingMovies', 'Contract', 'PaymentMethod']
GENDER_COLUMNS = ['gender']
BINARY_COLUMNS = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

binary_encoding = FunctionTransformer(lambda x: x.replace({'Yes': 1, 'No': 0}), feature_names_out='one-to-one')
gender_encoding = FunctionTransformer(lambda x: x.replace({'Male': 1, 'Female': 0}), feature_names_out='one-to-one')

preprocessor = ColumnTransformer([
    ('numerical', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), NUMERICAL_COLUMNS),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), CATEGORICAL_COLUMNS),
    ('binary', Pipeline([
        ('encoding', binary_encoding)
    ]), BINARY_COLUMNS),
    ('gender', Pipeline([
        ('encoding', gender_encoding)
    ]), GENDER_COLUMNS)
], remainder='drop')

preprocessor.fit(X_train)
X_train_pre = pd.DataFrame(preprocessor.transform(X_train), columns=preprocessor.get_feature_names_out(), index=X_train.index)
X_train_pre.head()

In [ ]:
from sklearn.metrics import silhouette_score
k_range = range(2, 6)
wcss, sil = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labs = km.fit_predict(X_train_pre.values)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_train_pre.values, labs))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(k_range, wcss, marker='o')
axes[0].set_title('Elbow')
axes[1].plot(k_range, sil, marker='o')
axes[1].set_title('Silhouette')
plt.tight_layout(); plt.show()

In [ ]:
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init='auto')
kmeans.fit(X_train_pre.values)
train_clusters = kmeans.predict(X_train_pre.values)
pd.Series(train_clusters).value_counts().sort_index()

In [ ]:
X_train_feat = X_train_pre.copy()
X_train_feat['cluster'] = train_clusters.astype(int)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_feat.values, y_train.values)
X_train_res = pd.DataFrame(X_train_res, columns=X_train_feat.columns)
pd.Series(y_train_res).value_counts()

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_res, y_train_res)

In [ ]:
X_test_pre = pd.DataFrame(preprocessor.transform(X_test), columns=preprocessor.get_feature_names_out(), index=X_test.index)
test_clusters = kmeans.predict(X_test_pre.values)
X_test_feat = X_test_pre.copy()
X_test_feat['cluster'] = test_clusters.astype(int)

y_pred = clf.predict(X_test_feat.values)
y_proba = clf.predict_proba(X_test_feat.values)[:, 1]

print(classification_report(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_proba))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=axes[0])
axes[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0,1],[0,1],'--')
axes[1].legend()
axes[1].set_title('ROC')
plt.tight_layout(); plt.show()